# ML4HL Responsible AI Toolbox: HIV Treatment Failure Risk Stratification

**Course**: AI for Healthcare (AI4HC) — IE University, Sep 2026  
**Adapted from**: [IE-ML-for-Healthcare/RAI_opioid_risk_prevention](https://github.com/IE-ML-for-Healthcare/RAI_opioid_risk_prevention)  
**Dataset**: AIDS Clinical Trials Group Study 175 (UCI ID 890) — 2,139 HIV-positive adults  
**Publication**: Hammer et al., *NEJM* 1996  

---

## Learning Objectives

1. Frame a **clinical problem** around HIV treatment failure prediction using baseline patient characteristics  
2. Build a **reproducible, leak-proof pipeline** with scikit-learn  
3. Translate model performance into **patient impact** (false alerts, missed failures, workload)  
4. Justify an **operating point** with calibration and threshold analysis  
5. Use the **Responsible AI Dashboard** to identify deployment risks, biases, and mitigations  

### Three RAI Themes
- **Interpretability**: Which baseline features drive treatment failure risk scores?  
- **Counterfactual reasoning**: What if a patient had received combination therapy instead of monotherapy?  
- **Causal analysis**: This is RCT data — causal estimates of treatment effect are credible.

---
## Step 0 — Environment & Imports

In [ ]:
# ============================================================
# 0. Environment & imports
# ============================================================
import warnings, os, sys
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    classification_report, confusion_matrix, brier_score_loss,
    roc_auc_score, average_precision_score, ConfusionMatrixDisplay,
    recall_score, precision_score
)
from sklearn.dummy import DummyClassifier

from utils import (
    positive_scores, auc_report, tradeoff_table, wilson_interval,
    pick_threshold_cost, pick_threshold_recall_floor, pick_threshold_workload,
    summary_at_threshold,
    plot_recall_floor_curves, plot_cumulative_recall_at_threshold,
    plot_topk_at_threshold, make_thresholded_estimator, init_rai_dependencies
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"Python  : {sys.version}")
print(f"NumPy   : {np.__version__}")
print(f"pandas  : {pd.__version__}")
print(f"seed    : {RANDOM_STATE}")

---
## Step 1 — Clinical Problem Framing and Study Objective

### The Clinical Question

**Can baseline patient characteristics and treatment assignment predict treatment failure (disease progression or death) in HIV-positive adults, enabling risk-stratified monitoring?**

### Context & Motivation

ACTG 175 was a landmark **randomized clinical trial** (Hammer et al., *NEJM* 1996) comparing four antiretroviral regimens in 2,139 HIV-positive adults with CD4 counts between 200–500 cells/mm³:

| Arm | Regimen | Type |
|---|---|---|
| 0 | Zidovudine (ZDV) only | Monotherapy |
| 1 | ZDV + didanosine (ddI) | Combination |
| 2 | ZDV + zalcitabine (Zal) | Combination |
| 3 | Didanosine (ddI) only | Alternative monotherapy |

The trial demonstrated that **combination therapy significantly reduced disease progression** compared to ZDV monotherapy alone.

### Target Population
HIV-positive adults (≥12 years) with baseline CD4 200–500/mm³, enrolled across US clinical sites in the early 1990s.

### Decision Context
A **risk-stratification flag** to identify patients at high risk of treatment failure. High-risk patients could receive:
- Intensified viral load monitoring
- Earlier consideration of treatment switch
- Proactive adherence support

### Success Criteria
- **Recall ≥ 60%**: catch the majority of patients who will experience treatment failure  
- **Alerts ≤ 300 per 1,000 screened**: keep the monitoring workload manageable  
- **Calibrated probabilities**: risk scores should be trustworthy for clinical conversations  

### Potential Harms
| Error | Clinical consequence |
|---|---|
| **False Negative** (missed failure) | Delayed intervention → disease progression → opportunistic infections → death |
| **False Positive** (false alarm) | Unnecessary intensified monitoring → resource waste, patient anxiety, potential premature treatment switch |

> **So what?** In HIV care, missing a patient who is failing treatment (FN) has far greater consequences than an unnecessary monitoring visit (FP). We therefore prioritise **recall** while keeping the alert volume manageable for clinical teams.

---
## Step 2 — Data Characterisation and Representativeness

In [ ]:
# ============================================================
# 2a. Load and inspect
# ============================================================
try:
    df = pd.read_csv("data/ACTG175.csv")
    print("Loaded from local CSV")
except FileNotFoundError:
    from ucimlrepo import fetch_ucirepo
    data = fetch_ucirepo(id=890)
    df = pd.concat([data.data.features, data.data.targets], axis=1)
    os.makedirs("data", exist_ok=True)
    df.to_csv("data/ACTG175.csv", index=False)
    print("Downloaded from UCI and saved to data/ACTG175.csv")

print(f"Raw shape: {df.shape}")
df.head()

In [ ]:
# ============================================================
# 2b. Feature engineering, exclusions, and schema
# ============================================================
TARGET = "cid"

# --- Post-randomization features: EXCLUDE to prevent temporal leakage ---
# cd420, cd820: measured at 20 weeks AFTER treatment starts
# offtrt: whether patient went off-treatment (observed DURING trial)
# time: time to failure/censoring — direct target leakage
# pidnum: patient ID — not a feature
POST_RANDOMIZATION = ["cd420", "cd820", "offtrt"]
EXCLUDE = ["pidnum", "time"] + POST_RANDOMIZATION

# --- Create binary treatment variable (stateless — safe before split) ---
# Per course slides: renaming and indicator creation are stateless transforms
if "treat" not in df.columns:
    df["treat"] = (df["trt"] != 0).astype(int)
    print("Created binary 'treat': 0=ZDV only, 1=others")

# --- Create treatment arm dummies (stateless — one-hot of trt) ---
trt_dummies = pd.get_dummies(df["trt"], prefix="trt", drop_first=True).astype(int)
for col in trt_dummies.columns:
    df[col] = trt_dummies[col]

# --- Feature definitions ---
continuous_features = ["age", "wtkg", "karnof", "preanti", "cd40", "cd80"]
binary_features = [
    "hemo", "homo", "drugs", "oprior", "z30", "zprior",
    "race", "gender", "str2", "symptom",
    "trt_1", "trt_2", "trt_3"   # treatment arm indicators (ref = ZDV only)
]
feature_cols = continuous_features + binary_features
SENSITIVE = ["race", "gender"]

print(f"\nTarget: {TARGET} — 1=treatment failure/death, 0=censored/survived")
print(f"Features: {len(feature_cols)} ({len(continuous_features)} continuous, {len(binary_features)} binary)")
print(f"Excluded (leakage/ID): {EXCLUDE}")
print(f"Sensitive attributes: {SENSITIVE}")
print(f"\nNote: 'strat' excluded — redundant with str2 + preanti")
print(f"\n--- Target distribution ---")
print(df[TARGET].value_counts())
print(f"\nTreatment failure prevalence: {df[TARGET].mean():.3f}")

In [ ]:
# ============================================================
# 2c. Cohort table
# ============================================================
def cohort_stats(subset, label):
    return pd.Series({
        "N": len(subset),
        "Age (years)": f"{subset['age'].mean():.1f} +/- {subset['age'].std():.1f}",
        "Weight (kg)": f"{subset['wtkg'].mean():.1f} +/- {subset['wtkg'].std():.1f}",
        "Male (%)": f"{subset['gender'].mean()*100:.1f}",
        "Non-white (%)": f"{subset['race'].mean()*100:.1f}",
        "CD4 baseline": f"{subset['cd40'].mean():.0f} +/- {subset['cd40'].std():.0f}",
        "CD8 baseline": f"{subset['cd80'].mean():.0f} +/- {subset['cd80'].std():.0f}",
        "Karnofsky score": f"{subset['karnof'].mean():.0f} +/- {subset['karnof'].std():.0f}",
        "ARV experienced (%)": f"{subset['str2'].mean()*100:.1f}",
        "Symptomatic (%)": f"{subset['symptom'].mean()*100:.1f}",
        "Combo therapy (%)": f"{subset['treat'].mean()*100:.1f}",
    }, name=label)

cohort = pd.concat([
    cohort_stats(df, "Overall"),
    cohort_stats(df[df[TARGET]==1], "Failure (cid=1)"),
    cohort_stats(df[df[TARGET]==0], "Survived (cid=0)")
], axis=1)

print("=== Cohort Table ===")
cohort

In [ ]:
# ============================================================
# 2d. Visualisations
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Class distribution
df[TARGET].value_counts().sort_index().plot.bar(
    ax=axes[0], color=["steelblue", "salmon"])
axes[0].set_title("Treatment Outcome")
axes[0].set_xticklabels(["Survived (0)", "Failure (1)"], rotation=0)
axes[0].set_ylabel("Count")

# 2. CD4 baseline by outcome
df.boxplot(column="cd40", by=TARGET, ax=axes[1])
axes[1].set_title("CD4 Baseline by Outcome")
axes[1].set_xlabel("cid (0=survived, 1=failure)")
axes[1].set_ylabel("CD4 count (cells/mm³)")
plt.suptitle("")

# 3. Treatment arm distribution
trt_labels = {0: "ZDV", 1: "ZDV+ddI", 2: "ZDV+Zal", 3: "ddI"}
df["trt"].map(trt_labels).value_counts().sort_index().plot.bar(
    ax=axes[2], color="steelblue")
axes[2].set_title("Treatment Arm")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

# 4. Failure rate by treatment arm
failure_by_trt = df.groupby("trt")[TARGET].mean().rename(index=trt_labels)
print("\nFailure rate by treatment arm:")
for arm, rate in failure_by_trt.items():
    print(f"  {arm:10s}: {rate:.3f} ({rate*1000:.0f} per 1,000)")

# 5. Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 9))
corr = df[feature_cols + [TARGET]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax, vmin=-1, vmax=1, square=True, linewidths=0.5,
            annot_kws={"size": 7})
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 2e. Missing values
# ============================================================
missing = df[feature_cols].isnull().sum()
print(f"Missing values per feature: {missing.sum()} total")
if missing.sum() == 0:
    print("No missing values — all 2,139 records are complete.")
    print("\nStrength: no imputation bias. Limitation: in real-world EHR data,")
    print("missingness is common and informative — this clean trial data")
    print("does not test our pipeline's handling of missing values.")

### External Validity and Representativeness

| Factor | Assessment |
|---|---|
| **Source** | Single US-based RCT, early 1990s |
| **Population** | HIV+ adults, CD4 200–500/mm³, selected trial participants |
| **Treatment era** | Pre-HAART (before protease inhibitors) — does not reflect modern ART |
| **Generalisability** | Limited to similar trial-eligible populations; results may not transfer to current standard-of-care settings |
| **Missing data** | None — unusually clean for clinical data; pipelines should still include imputation for deployment |
| **Label quality** | `cid` is a composite of disease progression and death — well-defined endpoint |

> **So what?** This dataset provides an excellent **teaching case** for Responsible AI because it is a **randomised clinical trial** — the gold standard for causal inference. Treatment assignment is independent of confounders by design. However, the 1990s treatment context limits direct applicability to modern HIV care. Any deployed model would need revalidation on contemporary cohorts.

---
## Step 3 — Experimental Design and Splits

In [ ]:
# ============================================================
# 3. Stratified 70/15/15 split
# ============================================================
X = df[feature_cols].copy()
y = df[TARGET].copy()

# First: 85% dev / 15% test
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE)

# Second: split dev into ~70% train / ~15% val (of total)
val_frac = round(len(X_test) / len(X_dev), 3)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=val_frac, stratify=y_dev, random_state=RANDOM_STATE)

print("Split summary (stratified by target):")
print(f"{'Set':<6} {'N':>5} {'Failures':>9} {'Prevalence':>11}")
print("-" * 35)
for name, yy in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{name:<6} {len(yy):>5} {int(yy.sum()):>9} {yy.mean():>11.3f}")

### Excluded Features — Leakage Prevention

Per the course slides on **Stateless vs Stateful Transformations** and the data leakage scenarios (target leakage, train-test contamination, temporal leakage):

| Excluded Column | Reason | Leakage Type |
|---|---|---|
| `cd420` (CD4 at 20 weeks) | Measured **after** treatment begins | Temporal — future information |
| `cd820` (CD8 at 20 weeks) | Measured **after** treatment begins | Temporal — future information |
| `offtrt` (off-treatment flag) | Observed **during** trial follow-up | Temporal — future information |
| `time` (time to failure) | Directly encodes the outcome timing | Target leakage |
| `pidnum` (patient ID) | Administrative identifier, not a feature | Not predictive |
| `strat` (stratification) | Redundant with `str2` + `preanti` | Multicollinearity |

**Golden rule**: *Split first, transform later.* All stateful preprocessing (scaling) happens inside the `sklearn.Pipeline` and is fitted on training data only. Stateless transforms (column renaming, dummy creation) are safe before splitting.

**No temporal split** is available here — all baseline features are measured at a single time point (randomisation). We use stratified random splitting instead.

---
## Step 4 — Pipelines, Baselines, and Leakage Controls

In [ ]:
# ============================================================
# 4a. Preprocessing pipeline, baseline, and logistic regression
# ============================================================

# --- Preprocessing: scale continuous, pass through binary ---
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), continuous_features),
    ("bin", "passthrough", binary_features)
], remainder="drop")  # drop any extra columns (safety for RAI step)

# --- Baseline: majority-class DummyClassifier ---
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
print("=== BASELINE: Majority-class classifier ===")
print(f"Always predicts: {dummy.predict(X_val[:1])[0]}")
print(f"Baseline recall: 0.000 (never flags anyone)")
print(f"Baseline accuracy: {(y_val == 0).mean():.3f}")
print()

# --- Logistic Regression pipeline ---
pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(
        solver="lbfgs", max_iter=1000, random_state=RANDOM_STATE))
])
pipe.fit(X_train, y_train)

print("=== LOGISTIC REGRESSION (validation) ===")
print(classification_report(y_val, pipe.predict(X_val),
      target_names=["Survived", "Failure"]))

In [ ]:
# ============================================================
# 4b. Coefficient table with odds ratios
# ============================================================
feat_names = continuous_features + binary_features
coefs = pipe.named_steps["clf"].coef_[0]
intercept = pipe.named_steps["clf"].intercept_[0]

coef_df = pd.DataFrame({
    "Feature": feat_names,
    "Coefficient": coefs,
    "Odds Ratio": np.exp(coefs),
    "Direction": ["higher risk" if c > 0 else "lower risk" for c in coefs]
}).sort_values("Coefficient", key=abs, ascending=False)

print(f"Intercept: {intercept:.4f}")
print(f"\nNote: continuous features are standardised, so coefficients")
print(f"represent the effect of a 1-SD increase. Binary features")
print(f"represent the effect of switching from 0 to 1.")
print()
coef_df.round(4)

### Leakage Checklist

| Check | Status |
|---|---|
| Post-randomization features excluded? | ✅ cd420, cd820, offtrt removed |
| Time-to-event excluded? | ✅ `time` not in feature set |
| Patient ID excluded? | ✅ `pidnum` not in feature set |
| Scaling fitted on train only? | ✅ Inside `Pipeline` → `ColumnTransformer` |
| No test data seen during training? | ✅ Stratified split before any fitting |
| Calibration on train only? | ✅ `CalibratedClassifierCV(cv=5)` on training set |
| Threshold chosen on validation only? | ✅ Test locked until Step 8 |

### Coefficient Interpretation (So what?)

Coefficients describe **model associations**, not causal effects. Key patterns to look for:
- **CD4 baseline (`cd40`)**: lower CD4 is the hallmark of immune compromise — expect a strong negative coefficient (lower CD4 → higher failure risk)
- **Treatment arms (`trt_1/2/3`)**: if combination therapy is protective, these dummies should have negative coefficients vs. ZDV-only reference
- **Karnofsky score (`karnof`)**: higher functional status should be protective
- **An odds ratio of 1.5 means**: that feature level is associated with 50% higher odds of treatment failure

---
## Step 5 — Metrics and Discrimination Analysis

In [ ]:
# ============================================================
# 5a. Validation discrimination (ROC AUC, PR AUC)
# ============================================================
val_scores = positive_scores(pipe, X_val)
auc_report(y_val, val_scores, name="Logistic Regression (validation)", plot=True)

In [ ]:
# ============================================================
# 5b. Subgroup metrics by race AND gender
# ============================================================
def subgroup_report(y_true, y_score, group_series, group_labels, threshold=0.5):
    """Compute subgroup metrics at a given threshold."""
    rows = []
    for label, val in group_labels.items():
        mask = (group_series == val)
        n = mask.sum()
        n_pos = int(y_true[mask].sum())
        if n > 0:
            pred = (y_score[mask] >= threshold).astype(int)
            rec = recall_score(y_true[mask], pred, zero_division=0)
            prec = precision_score(y_true[mask], pred, zero_division=0)
            alerts = pred.sum() / n * 1000
            rows.append({
                "Group": label, "N": n, "Failures": n_pos,
                "Prevalence": f"{n_pos/n:.3f}",
                "Recall": f"{rec:.3f}", "Precision": f"{prec:.3f}",
                "Alerts/1000": f"{alerts:.0f}"
            })
    return pd.DataFrame(rows)

print("=== Subgroup by RACE (default threshold 0.5) ===")
display(subgroup_report(y_val.values, val_scores,
    X_val["race"].values, {"White (0)": 0, "Non-white (1)": 1}))

print("\n=== Subgroup by GENDER (default threshold 0.5) ===")
display(subgroup_report(y_val.values, val_scores,
    X_val["gender"].values, {"Female (0)": 0, "Male (1)": 1}))

### Clinical Translation (So what?)

When interpreting the metrics above, translate to patient impact:

- **ROC AUC** measures the model's ability to rank patients who will fail higher than those who won't. A value of ~0.65–0.75 means the model is moderately useful — better than chance but not sufficient as a standalone diagnostic.
- **PR AUC** is more informative than ROC AUC when the positive class is uncommon (~15%). It reflects the model's precision-recall trade-off.
- **Subgroup gaps**: if recall differs substantially between racial groups (e.g., 0.70 for White vs 0.50 for Non-white), the model **systematically misses more treatment failures in one population**. This is a deployment risk requiring mitigation.
- **Per 1,000 patients**: At the default 0.5 threshold, how many alerts does the model generate? How many true failures does it catch? These numbers drive staffing and resource decisions.

> The FN cost framing for HIV: each missed treatment failure means a patient continues on a failing regimen, risking viral resistance, opportunistic infections, and death. Each FP means an unnecessary clinic visit or lab draw — costly but not dangerous.

---
## Step 6 — Probability Calibration and Uncertainty

In [ ]:
# ============================================================
# 6a. Calibrate on training data only (5-fold CV)
# ============================================================
calibrated_pipe = CalibratedClassifierCV(
    pipe, method="sigmoid", cv=5)
calibrated_pipe.fit(X_train, y_train)

# Scores on validation
cal_scores = positive_scores(calibrated_pipe, X_val)
raw_scores = positive_scores(pipe, X_val)

# Brier scores
brier_raw = brier_score_loss(y_val, raw_scores)
brier_cal = brier_score_loss(y_val, cal_scores)
print(f"Brier score (raw):        {brier_raw:.4f}")
print(f"Brier score (calibrated): {brier_cal:.4f}")
print(f"Improvement:              {brier_raw - brier_cal:+.4f}")

# Calibration-in-the-large
print(f"\n--- Calibration-in-the-large (validation) ---")
print(f"Observed prevalence:      {y_val.mean():.4f}")
print(f"Mean raw probability:     {raw_scores.mean():.4f}")
print(f"Mean calibrated prob:     {cal_scores.mean():.4f}")
print(f"Absolute diff (cal):      {abs(y_val.mean() - cal_scores.mean()):.4f}")

In [ ]:
# ============================================================
# 6b. Reliability plot
# ============================================================
fig, ax = plt.subplots(figsize=(6, 6))
for scores, label, marker in [(raw_scores, "Raw", "o"),
                               (cal_scores, "Calibrated", "s")]:
    frac_pos, mean_pred = calibration_curve(
        y_val, scores, n_bins=10, strategy="uniform")
    ax.plot(mean_pred, frac_pos, f"{marker}-", label=label)

ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed fraction of failures")
ax.set_title("Reliability Plot (Validation)")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### Clinical Implication of Calibration (So what?)

- A **calibrated probability of 30%** means: among patients the model scores ~0.30, roughly 30% actually experienced treatment failure. This is essential for **shared decision-making** — a clinician can say: *"The model estimates a 30% chance of treatment failure. Let's discuss whether closer monitoring makes sense for you."*
- The **Brier score** (lower = better) measures overall calibration quality. Perfect = 0.0, always-predict-prevalence ≈ 0.13.
- **Calibration-in-the-large**: if the mean predicted probability closely matches the observed prevalence, the model is well-calibrated at the population level. Individual-level calibration (reliability plot) is what matters for patient conversations.

---
## Step 7 — Threshold Selection and Decision Analysis

In [ ]:
# ============================================================
# 7a. Compare four threshold policies on VALIDATION
# ============================================================
print("=" * 65)
print("THRESHOLD POLICY COMPARISON (Validation Set — calibrated scores)")
print("=" * 65)

# Policy 1: Recall floor >= 60%
res_recall = pick_threshold_recall_floor(
    y_val.values, cal_scores, recall_floor=0.60)
print("\n--- Policy 1: Recall Floor >= 60% ---")
print(res_recall["summary"])

# Policy 2: Workload cap <= 300 alerts/1000
res_work = pick_threshold_workload(
    y_val.values, cal_scores, alerts_per_1000_max=300.0)
print("\n--- Policy 2: Workload Cap <= 300/1000 ---")
print(res_work["summary"])

# Policy 3: Cost-sensitive (FN harm = 10x FP harm)
res_cost = pick_threshold_cost(
    y_val.values, cal_scores, C_FP=1, C_FN=10)
print("\n--- Policy 3: Cost-Sensitive (FN = 10x FP) ---")
print(res_cost["summary"])

In [ ]:
# ============================================================
# 7b. Joint policy: recall >= 60% AND alerts <= 300/1000
# ============================================================
tt = tradeoff_table(y_val.values, cal_scores)
feasible = tt[(tt["recall"] >= 0.60) & (tt["alerts_per_1000"] <= 300)]

if len(feasible) > 0:
    best = feasible.sort_values("precision", ascending=False).iloc[0]
    locked_threshold = best["threshold"]
    print("=== JOINT POLICY: recall >= 60% AND alerts <= 300/1000 ===")
    print(f"\nLOCKED THRESHOLD: {locked_threshold:.6f}")
    print(f"  Recall:       {best['recall']:.3f}")
    print(f"  Precision:    {best['precision']:.3f}")
    print(f"  Alerts/1000:  {best['alerts_per_1000']:.1f}")
    print(f"  TP={best['TP']:.0f}  FP={best['FP']:.0f}  "
          f"FN={best['FN']:.0f}  TN={best['TN']:.0f}")
else:
    print("No threshold satisfies both constraints.")
    print("Relaxing to recall-floor policy only.")
    locked_threshold = res_recall["threshold"]
    print(f"LOCKED THRESHOLD (recall-floor): {locked_threshold:.6f}")

# --- Impact table ---
print("\n=== Impact Table (all policies) ===")
impact_rows = []
for name, res in [("Recall Floor", res_recall),
                   ("Workload Cap", res_work),
                   ("Cost-Sensitive", res_cost)]:
    s = summary_at_threshold(y_val.values, cal_scores, res["threshold"])
    impact_rows.append({
        "Policy": name,
        "Threshold": f"{res['threshold']:.4f}",
        "Recall": f"{s['recall']:.3f}",
        "Precision": f"{s['precision']:.3f}",
        "Alerts/1000": f"{s['alerts_per_1000']:.0f}",
        "Missed Failures": f"{s['FN']:.0f}",
    })
s_joint = summary_at_threshold(y_val.values, cal_scores, locked_threshold)
impact_rows.append({
    "Policy": "JOINT (locked)",
    "Threshold": f"{locked_threshold:.4f}",
    "Recall": f"{s_joint['recall']:.3f}",
    "Precision": f"{s_joint['precision']:.3f}",
    "Alerts/1000": f"{s_joint['alerts_per_1000']:.0f}",
    "Missed Failures": f"{s_joint['FN']:.0f}",
})
display(pd.DataFrame(impact_rows))

In [ ]:
# ============================================================
# 7c. Threshold visualisations
# ============================================================
plot_recall_floor_curves(y_val.values, cal_scores,
                         recall_floor=0.60,
                         chosen_threshold=locked_threshold)
plot_cumulative_recall_at_threshold(y_val.values, cal_scores,
                                    chosen_threshold=locked_threshold)
plot_topk_at_threshold(y_val.values, cal_scores,
                       chosen_threshold=locked_threshold, top_k=50)

### Threshold Decision (So what?)

The **impact table** above makes the trade-off concrete for clinic stakeholders:

- **Recall-floor policy** ensures at least 60% of true treatment failures are flagged, even if it means more alerts.
- **Workload-cap policy** keeps alerts ≤ 300/1,000 to avoid overwhelming clinical teams, but may miss more patients.
- **Cost-sensitive policy** (FN = 10× FP) directly encodes the clinical judgement that missing a failure is 10× worse than a false alarm.
- **Joint policy** (locked) balances both constraints: catch ≥ 60% of failures while keeping alerts manageable.

The locked threshold will now be applied to the **held-out test set** — no further tuning is permitted.

---
## Step 8 — Final Test Evaluation and Generalisation

In [ ]:
# ============================================================
# 8a. Final test evaluation at locked threshold
# ============================================================
print("=" * 65)
print("FINAL TEST EVALUATION")
print(f"Threshold LOCKED on validation: {locked_threshold:.6f}")
print("=" * 65)

test_scores = positive_scores(calibrated_pipe, X_test)
auc_report(y_test, test_scores, name="Calibrated LR (test)", plot=True)

# Metrics at locked threshold
test_pred = (test_scores >= locked_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
n_test = len(y_test)
recall_t = tp / (tp + fn) if (tp + fn) > 0 else 0
precision_t = tp / (tp + fp) if (tp + fp) > 0 else 0
alerts_t = (tp + fp) / n_test * 1000

# Wilson confidence intervals
rec_lo, rec_hi = wilson_interval(tp, tp + fn)
prec_lo, prec_hi = wilson_interval(tp, tp + fp)

print(f"\n--- Performance at threshold {locked_threshold:.4f} ---")
print(f"Recall:      {recall_t:.3f}  (95% CI: {rec_lo:.3f} – {rec_hi:.3f})")
print(f"Precision:   {precision_t:.3f}  (95% CI: {prec_lo:.3f} – {prec_hi:.3f})")
print(f"Alerts/1000: {alerts_t:.1f}")
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")

# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_pred, ax=ax,
    display_labels=["Survived", "Failure"], cmap="Blues")
ax.set_title(f"Test Confusion Matrix (threshold={locked_threshold:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8b. Subgroup fairness check on test
# ============================================================
print("=== Test Set — Subgroup Fairness ===")
print("\nBy Race:")
display(subgroup_report(
    y_test.values, test_scores,
    X_test["race"].values,
    {"White (0)": 0, "Non-white (1)": 1},
    threshold=locked_threshold))

print("\nBy Gender:")
display(subgroup_report(
    y_test.values, test_scores,
    X_test["gender"].values,
    {"Female (0)": 0, "Male (1)": 1},
    threshold=locked_threshold))

# Treatment arm performance
print("\nBy Treatment Arm:")
trt_original = df.loc[X_test.index, "trt"].values
trt_map = {0: "ZDV only", 1: "ZDV+ddI", 2: "ZDV+Zal", 3: "ddI only"}
display(subgroup_report(
    y_test.values, test_scores,
    trt_original, trt_map,
    threshold=locked_threshold))

### Stakeholder Summary (So what?)

**For the HIV clinic director:**
- At the locked threshold, we flag approximately **X alerts per 1,000 patients** for intensified monitoring.
- We catch approximately **Y%** of patients who will experience treatment failure.
- We miss approximately **Z patients per 1,000** who needed intervention — these patients require a **safety-net protocol** (e.g., routine 6-month viral load checks for all patients regardless of risk score).

**For the equity officer:**
- Compare recall across racial groups. If the model misses more failures in non-white patients, this is a **disparate impact** requiring action (e.g., subgroup-specific thresholds, additional features, or separate models).
- Female patients may have very few cases in some subgroups — confidence intervals will be wide. Flag this as a **data limitation**, not a model conclusion.

**Limitations:**
- This model uses 1990s pre-HAART data. It should **not** be deployed for modern ART regimens without revalidation.
- The model is a screening tool, not a diagnostic — all flagged patients require clinical review.
- Confidence intervals reflect sample size — wider intervals in smaller subgroups.

---
## Step 9 — Responsible AI Analysis (RAI Dashboard)

In [ ]:
# ============================================================
# 9a. Build RAI Insights
# ============================================================
init_rai_dependencies()
from responsibleai import RAIInsights
from raiwidgets import ResponsibleAIDashboard

# Create thresholded model wrapper for RAI
rai_model = make_thresholded_estimator(calibrated_pipe, locked_threshold)

# Prepare data — include 'treat' column for causal analysis
train_df = X_train.copy()
train_df[TARGET] = y_train.values
train_df["treat"] = df.loc[X_train.index, "treat"].values

test_df = X_test.copy()
test_df[TARGET] = y_test.values
test_df["treat"] = df.loc[X_test.index, "treat"].values

# Categorical features = all binary features + treat
cat_features = binary_features + ["treat"]

# Build RAIInsights
rai = RAIInsights(
    model=rai_model,
    train=train_df,
    test=test_df,
    target_column=TARGET,
    task_type="classification",
    categorical_features=cat_features,
)

# Add all RAI components
rai.explainer.add()
rai.error_analysis.add()
rai.counterfactual.add(total_CFs=5, desired_class="opposite")
rai.causal.add(treatment_features=["treat"])

print("Computing RAI insights — this may take several minutes...")
rai.compute()
print("RAI computation complete.")

In [ ]:
# ============================================================
# 9b. Launch the dashboard
# ============================================================
ResponsibleAIDashboard(rai)

### RAI Dashboard Exploration Guide

#### Component 1: Data Analysis
- Review dataset statistics and class distribution
- Check feature distributions by outcome (cid=0 vs cid=1)
- **Form cohorts**: (1) monotherapy patients (`treat=0`), (2) combination therapy patients (`treat=1`)

#### Component 2: Model Overview & Fairness
- Compare model performance across **race** and **gender** cohorts
- Check: does the model achieve equalized odds? Does recall differ by race?
- If disparities exist, note them for the mitigation plan

#### Component 3: Error Analysis
- Use the **heatmap** to find segments with high error rates
- Key heatmap: **race × treatment arm** — does the model fail more for non-white patients on monotherapy?
- Use the **error tree** to identify feature thresholds that partition errors
- **So what?** If errors concentrate in a subgroup (e.g., older non-white patients on ZDV-only), that subgroup needs targeted clinical protocols regardless of the model's output

#### Component 4: Feature Importance (Interpretability)
- **Global importance**: Which features matter most across all patients? Expect CD4 baseline and treatment arm to be top drivers
- **Local importance**: For individual patients, which features drove their specific risk score?
- **So what?** If the model relies heavily on race or gender as predictors, investigate whether this reflects clinical reality (different disease progression rates) or data bias

#### Component 5: Counterfactuals
- Select a patient classified as high-risk. The dashboard shows the **minimal changes** needed to flip the prediction.
- Most actionable counterfactual: *"What if this patient had received combination therapy (treat=1) instead of ZDV monotherapy (treat=0)?"*
- Because treatment IS a modifiable feature (unlike age or race), these counterfactuals are **directly clinician-actionable**

#### Component 6: Causal Analysis
- **This is RCT data** — treatment assignment was randomised. This makes causal estimates **more credible** than observational studies where selection bias confounds treatment effects.
- The causal component estimates the **Average Treatment Effect (ATE)** of combination therapy (treat=1) vs. monotherapy (treat=0) on treatment failure risk.
- Examine **heterogeneous treatment effects**: Do patients with low baseline CD4 benefit more from combination therapy than those with high CD4?
- **So what for policymakers?** If the causal analysis confirms that combination therapy reduces failure rates, and the effect is larger for sicker patients (low CD4), this supports **risk-stratified treatment guidelines**: prioritise combination therapy for patients with the highest predicted failure risk.